### page6.[실습] checkpointer로 기억하는 Agent 만들기

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import os
from dotenv import load_dotenv
# .env 파일에서 OPENAI_API_KEY 불러오기
load_dotenv()

#LLM 생성
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

# [1번 문제]
# InMemorySaver()로 checkpointer를 만들고, get_weather 하나만 tools로
# 갖는 agent를 checkpointer=checkpointer와 함께 만드세요.

@tool
def get_weather(city: str) -> str:
    """도시 이름을 받아 날씨 정보를 반환합니다."""
    weather_db = {"서울": "맑음, 23도"}
    return weather_db.get(city, "정보 없음")

# 대화기록을 관리할 객체
checkpointer = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[get_weather],
    checkpointer=checkpointer,
    system_prompt="당신은 사용자와의 이전 대화 내용을 잘 기억하고 반영하여 답하는 친절한 AI 어시스턴트입니다."
)
# [2번 문제]
# config = {"configurable": {"thread_id": "user-1"}} 를 만들고,
# agent.invoke({"messages": [{"role": "user", "content": "안녕! 내 이름은
# 지훈이야"}]}, config=config) 를 실행하세요.

config = {"configurable": {"thread_id": "user-1"}}

result = agent.invoke({"messages": [{"role": "user", 
                            "content": "안녕! 내 이름은 지훈이야"
                          }]}, 
             config=config
            )
print(result["messages"][-1].content)
# [도전 문제]
# 같은 config로 "내 이름이 뭐라고 했지?" 를 다시 invoke()해서,
# 답변에 방금 알려준 이름이 그대로 들어있는지 확인하세요.
result = agent.invoke({"messages": [{"role": "user", 
                            "content": "내 이름이 뭐라고 했지?"
                          }]}, 
             config=config
            )
print(result["messages"][-1].content)



안녕, 지훈아! 만나서 반가워. 오늘은 어떤 이야기를 하고 싶니?
네 이름은 지훈이야! 어떻게 도와줄까?


### page9.[실습] 여러 thread_id로 대화 분리하기

In [12]:
# 앞 슬라이드의 checkpointer가 달린 agent를 그대로 사용합니다.

# [1번 문제]
# thread_id가 서로 다른 config_a, config_b 두 개를 만드세요.
config_a = {"configurable":{"thread_id": "user-a"}}
config_b = {"configurable":{"thread_id": "user-b"}}

# [2번 문제]
# config_a로 "내 이름은 지훈이야" -> "내 이름이 뭐라고 했지?" 순서로
# 두 번 invoke()해서, 두 번째 답변에 이름이 들어있는지 확인하세요.
result = agent.invoke({"messages": [{"role": "user", 
                            "content": "내 이름은 지훈이야"
                          }]}, 
             config=config_a
            )
print(result["messages"][-1].content)
result = agent.invoke({"messages": [{"role": "user", 
                            "content": "내 이름이 뭐라고 했지?"
                          }]}, 
             config=config_a
            )
print(result["messages"][-1].content)
# [도전 문제]
# 이번엔 config_b로 "내 이름이 뭐라고 했지?" 를 바로 invoke()해서,
# config_a와 답변이 어떻게 다른지 비교하세요.
result = agent.invoke({"messages": [{"role": "user", 
                            "content": "내 이름이 뭐라고 했지?"
                          }]}, 
             config=config_b
            )
print(result["messages"][-1].content)


안녕하세요, 지훈님! 반갑습니다. 어떻게 도와드릴까요?
지훈님이라고 하셨습니다! 다른 궁금한 점이나 요청이 있으신가요?
죄송하지만, 이전 대화에서 당신의 이름을 언급하지 않으셨기 때문에 기억하고 있지 않습니다. 이름을 말씀해 주시면 앞으로 더 쉽게 대화할 수 있을 것 같습니다!


### page12.[실습] 여러 턴 대화 이어가며 확인하기

In [17]:
import ast, operator
# get_weather, calculate 두 Tool과 checkpointer를 가진 agent를 쓰세요.
_OPS = {
ast.Add: operator.add,      # + 기호
ast.Sub: operator.sub,      # - 기호
ast.Mult: operator.mul,     # * 기호
ast.Div: operator.truediv,  # / 기호
ast.USub: operator.neg,     # 음수 부호 (-)
ast.UAdd: operator.pos      # 양수 부호 (+)
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        return node.value
    if isinstance(node, ast.BinOp):
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp):
        return _OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError("허용되지 않은 수식입니다.")

@tool
def calculate(expression: str) -> str:
    """사칙연산 수식 문자열을 계산합니다. 예: '3 + 5 * 2'"""
    try:
        tree = ast.parse(expression, mode="eval")
        result = _safe_eval(tree.body)
        return f"{expression} = {result}"
    except Exception:
        return "계산할 수 없는 수식입니다."

# get_weather는 위에 선언한 함수로 대체

checkpointer = InMemorySaver()

agent = create_agent(
    model = llm,
    tools = [get_weather, calculate],
    checkpointer=checkpointer,
    system_prompt="당신은 사용자와의 이전 대화 내용을 잘 기억하고 반영하여 답하는 친절한 AI 어시스턴트입니다."
)

def ask_agent(query: str,config) -> str:
    result=agent.invoke({"messages": [
    {"role": "user", "content": query}]},config = config)
    print(len(result["messages"]))
    print(result["messages"])
    return result["messages"][-1].content

# [1번 문제]
# 하나의 thread_id로 아래 세 문장을 순서대로 invoke()하세요.
# "안녕! 내 이름은 지훈이야" -> "부산 날씨 어때?" -> "내 이름이 뭐라고 했지?"
config = {"configurable":{"thread_id": "user-1"}}

print(ask_agent("안녕! 내이름은 지훈이야", config))
print(ask_agent("서울 날씨 어때?", config))
print(ask_agent("내 이름이 뭐라고 했지?", config))

# [2번 문제]
# 세 번의 invoke() 결과마다 len(result["messages"])를 출력해서,
# 메시지 개수가 계속 늘어나는지 확인하세요.

# [도전 문제]
# 세 번째 답변에 "부산" 관련 내용과 "지훈" 이라는 이름이 모두
# 잘 반영되어 있는지, 즉 Tool 결과와 대화 기억이 함께 동작하는지 확인하세요.
# 따로 result["messages"] 출력해서 확인해보면 됨
# ToolMessage(content='맑음, 20도', name='get_weather')

2
[HumanMessage(content='안녕! 내이름은 지훈이야', additional_kwargs={}, response_metadata={}, id='db418c4d-7530-4a4f-8c81-be660cc48988'), AIMessage(content='안녕하세요, 지훈님! 만나서 반갑습니다. 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 122, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_242228cf5a', 'id': 'chatcmpl-EdRQZ3sEPSgvEw4D8mdNt0BUNbl1W', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17b3c-95b4-7790-a2fd-7d18a0765292-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 122, 'output_tokens': 22, 'total_tokens': 144, 'input_token_details': {'audio': 0, 'cache_re

In [ ]:
# 지난 시간에 만든 get_weather, calculate, search_notice 세 Tool을 모두 사용합니다.

# 3. 임시 데이터(문서) 준비 및 벡터 저장소 생성
docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
]

#RAG 구현: 검색 증강 생성  --- agent가 질문과 관련이 있는 내용을 찾아서 답변으로 생성
#vectorstrore 생성
#retriever 생성

# 기억 저장소: InMemorySaver 생성


# [1번 문제]
# tools=[get_weather, calculate, search_notice] 와 checkpointer=checkpointer를
# 모두 가진 agent를 만드세요.

# [2번 문제]
# agent, thread_id, question을 받아 agent.invoke()의 최종 답변을 반환하는
# ask_with_memory(agent, thread_id, question) 함수를 작성하세요.

# [도전 문제]
# 같은 thread_id로 "내 이름은 지훈이야" -> "수료증은 어떻게 받나요?" ->
# "내 이름이 뭐라고 했지?" 를 순서대로 실행해, Tool·검색·기억이 한 대화 안에서
# 함께 잘 동작하는지 확인하세요.
